# Datasets Sources
- https://www.kaggle.com/competitions/demand-forecasting-kernels-only/models
- https://www.kaggle.com/competitions/m5-forecasting-accuracy/data


# Model
- LSTM, GRU, RNN comparison

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import copy
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# Hyperparameter
DATA_DIR = Path("/content")  # Colab: upload the two CSV files here
BATCH_SIZE = 64
EPOCHS = 15
learning_rate = 1e-3
SEED = 7
LOOKBACK = 28
HIDDEN_SIZE = 32
HOLDOUT = 28

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
print("Device:", device, "| PyTorch:", torch.__version__)

In [ ]:
# Build M5 Time-Series Pipeline
day_cols = pd.read_csv(DATA_DIR / "sales_train_evaluation.csv", nrows=0).columns
day_cols = sorted([c for c in day_cols if c.startswith("d_")], key=lambda c: int(c[2:]))
sales = np.zeros(len(day_cols), dtype=np.float64)
for chunk in pd.read_csv(DATA_DIR / "sales_train_evaluation.csv",
                         usecols=day_cols, dtype=np.int32, chunksize=2000):
    sales += chunk[day_cols].sum(axis=0).to_numpy()

calendar = pd.read_csv(DATA_DIR / "calendar.csv", usecols=["d", "date"])
dates = pd.to_datetime(calendar.set_index("d").loc[day_cols, "date"]).to_numpy()
train_end = len(sales) - 2 * HOLDOUT
val_end = len(sales) - HOLDOUT
for name, start, end in [("Train", 0, train_end),
                          ("Validation", train_end, val_end),
                          ("Test", val_end, len(sales))]:
    print(name, str(dates[start])[:10], "to", str(dates[end - 1])[:10], end - start, "days")

In [ ]:
# Scaling and normalization (fit on Train ONLY)
mu = sales[:train_end].mean()
sigma = max(sales[:train_end].std(), 1e-8)
scaled = ((sales - mu) / sigma).astype(np.float32)
X = np.stack([scaled[t - LOOKBACK:t] for t in range(LOOKBACK, len(sales))])
y = scaled[LOOKBACK:]
target_days = np.arange(LOOKBACK, len(sales))

def make_set(mask):
    return TensorDataset(torch.from_numpy(X[mask, :, None]), torch.from_numpy(y[mask]))

train_set = make_set(target_days < train_end)
val_set = make_set((target_days >= train_end) & (target_days < val_end))
test_set = make_set(target_days >= val_end)

In [ ]:
# Build Dataloader
def make_train_loader():
    return DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                      generator=torch.Generator().manual_seed(SEED))

val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)
inputs, targets = next(iter(val_loader))
print("Input:", tuple(inputs.shape), "Target:", tuple(targets.shape))

In [ ]:
# Build Model
class ForecastRNN(nn.Module):
    def __init__(self, kind):
        super().__init__()
        self.recurrent = getattr(nn, kind)(1, HIDDEN_SIZE, batch_first=True)
        self.regressor = nn.Linear(HIDDEN_SIZE, 1)

    def forward(self, x):
        sequence, _ = self.recurrent(x)
        return self.regressor(sequence[:, -1]).squeeze(-1)

In [ ]:
# Inspect shapes
with torch.no_grad():
    for kind in ["LSTM", "GRU", "RNN"]:
        model = ForecastRNN(kind).to(device)
        print(kind, tuple(inputs.shape), "->", tuple(model(inputs.to(device)).shape))

In [ ]:
# Define loss and optimizer
criterion = nn.MSELoss()
models, optimizers = {}, {}
for kind in ["LSTM", "GRU", "RNN"]:
    torch.manual_seed(SEED)
    models[kind] = ForecastRNN(kind).to(device)
    optimizers[kind] = torch.optim.Adam(models[kind].parameters(), lr=learning_rate)

In [ ]:
# Training and Validating
histories, rows = {}, []

for kind in ["LSTM", "GRU", "RNN"]:
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    model = models[kind]
    optimizer = optimizers[kind]
    train_loader = make_train_loader()
    history = {"train_loss": [], "val_loss": []}
    best_val_loss, best_epoch, best_state = float("inf"), 0, None
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * len(y)

        model.eval()
        val_loss_sum = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                val_loss_sum += criterion(model(x), y).item() * len(y)

        train_loss = train_loss_sum / len(train_set)
        val_loss = val_loss_sum / len(val_set)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        if val_loss < best_val_loss:
            best_val_loss, best_epoch = val_loss, epoch
            best_state = copy.deepcopy(model.state_dict())
        print(f"{kind} Epoch {epoch:02}/{EPOCHS} | train {train_loss:.4f} | val {val_loss:.4f}")

    if device.type == "cuda":
        torch.cuda.synchronize()
    train_seconds = time.perf_counter() - start
    model.load_state_dict(best_state)
    model.eval()
    models[kind] = model
    histories[kind] = history
    rows.append({"Model": kind, "Best_epoch": best_epoch,
                 "Val_RMSE": np.sqrt(best_val_loss) * sigma,
                 "Parameters": sum(p.numel() for p in model.parameters()),
                 "Train_seconds": train_seconds})
    print(f"Restored {kind} epoch {best_epoch}: val loss = {best_val_loss:.4f}")

In [ ]:
# Plot learning curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, row in zip(axes, rows):
    kind = row["Model"]
    epochs = np.arange(1, EPOCHS + 1)
    ax.plot(epochs, histories[kind]["train_loss"], label="Training")
    ax.plot(epochs, histories[kind]["val_loss"], label="Validation")
    ax.axvline(row["Best_epoch"], color="gray", linestyle="--", label="Selected epoch")
    ax.set(title=kind, xlabel="Epoch", ylabel="MSE (normalized)")
    ax.legend()
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluation
y_true = sales[val_end:]
predictions = {}
selected = min(rows, key=lambda row: row["Val_RMSE"])["Model"]
for row in rows:
    kind = row["Model"]
    with torch.no_grad():
        pred = torch.cat([models[kind](x.to(device)).cpu() for x, _ in test_loader]).numpy()
    pred = pred * sigma + mu
    predictions[kind] = pred
    row["Test_MAE"] = np.mean(np.abs(y_true - pred))
    row["Test_RMSE"] = np.sqrt(np.mean((y_true - pred) ** 2))

comparison = pd.DataFrame(rows).set_index("Model")
comparison = comparison[["Best_epoch", "Val_RMSE", "Test_MAE", "Test_RMSE", "Parameters", "Train_seconds"]]
print(comparison.round(3).to_string())
print("Selected by validation:", selected)
print("Lowest test RMSE in this run:", comparison["Test_RMSE"].idxmin())

# Weekly seasonal-naive reference: use observed sales seven days earlier.
naive = sales[val_end - 7:-7]
print(f"Seasonal naive | Test MAE {np.mean(np.abs(y_true - naive)):.3f} | "
      f"Test RMSE {np.sqrt(np.mean((y_true - naive) ** 2)):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].plot(dates[val_end:], y_true, "k-o", label="Actual", markersize=3)
for kind, pred in predictions.items():
    axes[0].plot(dates[val_end:], pred, label=kind)
axes[0].set(title="Test: rolling one-step forecast", ylabel="Total units / day")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend()
comparison[["Test_MAE", "Test_RMSE"]].plot.bar(ax=axes[1], rot=0)
axes[1].set(title="Test error (lower is better)", ylabel="Total units / day")
plt.tight_layout()
plt.show()